In [ ]:
# ── Reproducibilidad global ─────────────────────────────────────────
import random, os, gzip
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
# ── Carga del dataset (compatible con Colab y local) ────────────────
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    os.system("git clone https://github.com/Nicolascresp/Airbnb.git 2>/dev/null || echo 'ya clonado'")
    DATA_PATH     = "Airbnb/listings.csv"
    CALENDAR_PATH = "Airbnb/calendar.csv.gz"
else:
    DATA_PATH     = os.path.join(os.path.dirname(os.path.abspath("__file__")), "listings.csv")
    CALENDAR_PATH = os.path.join(os.path.dirname(os.path.abspath("__file__")), "calendar.csv.gz")

df = pd.read_csv(DATA_PATH)
print(f"✅ Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

## 1. Descripción inicial del dataset

El dataset utilizado corresponde a publicaciones de Airbnb en Madrid, descargado desde Inside Airbnb. Cada fila representa un alojamiento publicado en la plataforma.

El dataset cuenta con 25.000 observaciones y 18 variables. Incluye información sobre ubicación, tipo de habitación, precio, cantidad de noches mínimas, reviews, disponibilidad anual y cantidad de publicaciones del host.

El problema de Machine Learning elegido será de regresión, con el objetivo de predecir el precio de un alojamiento (`price`) a partir de sus características.

In [ ]:
nulos = pd.DataFrame({
    "nulos": df.isnull().sum(),
    "porcentaje": (df.isnull().sum() / len(df) * 100).round(2)
})

nulos.sort_values("porcentaje", ascending=False)

### Análisis de valores faltantes

Se observan valores faltantes principalmente en `license`, `price`, `reviews_per_month` y `last_review`.

La variable `price` tiene 6.047 valores faltantes, lo que representa el 24,19% del dataset. Como `price` será la variable objetivo del modelo, no corresponde imputarla, ya que estaríamos inventando el valor que queremos predecir. Por eso, para el modelado se eliminarán las filas sin precio.

La variable `reviews_per_month` tiene valores faltantes en el 20,59% de los casos. Esto probablemente se deba a alojamientos sin reviews, por lo que puede interpretarse como ausencia de actividad y se podría imputar con 0.

La variable `last_review` también tiene 20,59% de faltantes y está relacionada con la ausencia de reviews. En lugar de usarla directamente, podría transformarse en una variable binaria que indique si el alojamiento tuvo alguna review.

La variable `license` tiene 63,25% de valores faltantes, por lo que inicialmente no parece recomendable usarla como variable predictora principal.

### Clasificación de valores faltantes

Los valores faltantes de `reviews_per_month` y `last_review` parecen ser MNAR, ya que probablemente no faltan al azar sino porque el alojamiento no recibió reviews. Es decir, el faltante tiene significado propio: ausencia de actividad registrada.

Los valores faltantes de `license` también pueden considerarse MNAR, porque la falta de licencia puede estar relacionada con decisiones del anfitrión, regulación o carga incompleta de información.

En el caso de `price`, al ser la variable objetivo, no se imputan los faltantes. Estos registros se eliminan para el modelado, ya que imputar el target implicaría inventar el valor que se busca predecir.

In [ ]:
df["price"].describe()

In [ ]:
df["price"].isnull().sum()

In [ ]:
(df["price"] == 0).sum()

In [ ]:
df_model = df.dropna(subset=["price"]).copy()

df_model.shape

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(df_model["price"], bins=50)
plt.title("Distribución de precios")
plt.xlabel("Precio")
plt.ylabel("Cantidad de alojamientos")
plt.show()

In [ ]:
df_model["price"].quantile([0, 0.25, 0.5, 0.75, 0.90, 0.95, 0.99, 1])

### Análisis inicial de la variable objetivo: `price`

La variable `price` será utilizada como target del modelo de regresión. Luego de eliminar los registros sin precio, quedan 18.953 observaciones disponibles para el análisis.

La distribución de precios muestra una fuerte asimetría hacia la derecha. La mediana es 110, mientras que la media es 156,69, lo que indica que algunos valores extremos elevan el promedio. Además, el percentil 99 es 793,36, pero el valor máximo llega a 25.654. Esto sugiere la presencia de outliers muy altos.

Por este motivo, no conviene analizar el precio solo con la media. Será necesario tratar los valores extremos o aplicar una transformación logarítmica para reducir el efecto de los outliers.

In [ ]:
df_model_filtrado = df_model[df_model["price"] <= df_model["price"].quantile(0.99)].copy()

plt.figure(figsize=(8,5))
plt.hist(df_model_filtrado["price"], bins=50)
plt.title("Distribución de precios hasta percentil 99")
plt.xlabel("Precio")
plt.ylabel("Cantidad de alojamientos")
plt.show()

### Distribución de precios sin valores extremos

Para visualizar mejor la distribución de precios, se generó un histograma excluyendo los valores por encima del percentil 99. Esta decisión permite observar el comportamiento de la mayoría de los alojamientos sin que los valores extremos distorsionen el gráfico.

La distribución sigue mostrando asimetría hacia la derecha: la mayoría de los alojamientos se concentra en precios bajos y medios, mientras que hay menos casos con precios altos.

In [ ]:
plt.figure(figsize=(8,5))
plt.boxplot(df_model_filtrado["price"], vert=False)
plt.title("Boxplot de precios hasta percentil 99")
plt.xlabel("Precio")
plt.show()

### Outliers en precio

El boxplot permite identificar valores atípicos en la variable `price`. Incluso luego de excluir los valores más extremos, se observan alojamientos con precios superiores al comportamiento típico.

Esto confirma que `price` presenta una distribución sesgada y que será necesario considerar el tratamiento de outliers o una transformación logarítmica antes de entrenar modelos predictivos.

In [ ]:
df_model_filtrado["room_type"].value_counts()

### Distribución por tipo de habitación

Se analiza la variable `room_type` para entender qué tipos de alojamientos predominan en el dataset. Esta variable es relevante porque el tipo de alojamiento puede influir directamente en el precio.

In [ ]:
plt.figure(figsize=(8,5))
df_model_filtrado["room_type"].value_counts().plot(kind="bar")
plt.title("Cantidad de alojamientos por tipo de habitación")
plt.xlabel("Tipo de habitación")
plt.ylabel("Cantidad de alojamientos")
plt.xticks(rotation=45)
plt.show()

El gráfico muestra la cantidad de alojamientos según el tipo de habitación. Esta visualización permite identificar qué categorías son más frecuentes dentro del dataset y ayuda a interpretar luego las diferencias de precio entre tipos de alojamiento.

In [ ]:
plt.figure(figsize=(8,5))
df_model_filtrado.boxplot(column="price", by="room_type")
plt.title("Precio por tipo de habitación")
plt.suptitle("")
plt.xlabel("Tipo de habitación")
plt.ylabel("Precio")
plt.xticks(rotation=45)
plt.show()

In [ ]:
df_model_filtrado.groupby("room_type")["price"].describe().round(2)

### Precio por tipo de habitación

El tipo de habitación muestra diferencias claras en el precio. Las categorías `Entire home/apt` y `Hotel room` presentan los precios típicos más altos, con medianas de 128 y 152 respectivamente. Sin embargo, la categoría `Hotel room` tiene muy pocas observaciones, por lo que su interpretación debe tomarse con cautela.

Las habitaciones privadas y compartidas presentan precios considerablemente menores, con medianas de 50 y 29,5. Esto tiene sentido desde el negocio, ya que alquilar una propiedad completa o una habitación de hotel suele ofrecer mayor privacidad y valor percibido que una habitación compartida.

Por este motivo, `room_type` será una variable relevante para el modelo predictivo.

In [ ]:
df_model_filtrado["neighbourhood"].value_counts().head(15)

In [ ]:
plt.figure(figsize=(10,5))
df_model_filtrado["neighbourhood"].value_counts().head(15).plot(kind="bar")
plt.title("Top 15 barrios con más alojamientos")
plt.xlabel("Barrio")
plt.ylabel("Cantidad de alojamientos")
plt.xticks(rotation=45, ha="right")
plt.show()

### Precio Promedio por Barrio (Histograma)

Para visualizar la distribución de precios promedio por barrio, calcularemos el precio promedio para cada barrio y luego mostraremos un histograma de estos promedios. Esto nos ayudará a entender mejor la variabilidad de precios a nivel de barrio y confirmar si `neighbourhood` es una variable útil.

In [ ]:
import seaborn as sns
# Calcular el precio promedio por barrio
avg_price_per_neighbourhood = df_model_filtrado.groupby('neighbourhood')['price'].mean().sort_values(ascending=False)

plt.figure(figsize=(12, 6))
sns.histplot(avg_price_per_neighbourhood, bins=20, kde=True)
plt.title('Distribución del Precio Promedio por Barrio')
plt.xlabel('Precio Promedio')
plt.ylabel('Cantidad de Barrios')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

print("Top 10 barrios con el precio promedio más alto:")
display(avg_price_per_neighbourhood.head(10))

### Distribución de alojamientos por barrio

El gráfico muestra que la oferta de alojamientos no está distribuida de manera uniforme entre los barrios de Madrid. Los barrios con mayor cantidad de publicaciones son `Embajadores`, `Universidad`, `Palacio`, `Sol`, `Justicia` y `Cortes`.

Esto sugiere una concentración importante de alojamientos en zonas céntricas y turísticas. Desde el punto de vista del modelo, `neighbourhood` puede ser una variable relevante para explicar diferencias de precio, ya que la ubicación suele influir directamente en el valor de un alojamiento.

In [ ]:
top_barrios = df_model_filtrado["neighbourhood"].value_counts().head(10).index

df_top_barrios = df_model_filtrado[df_model_filtrado["neighbourhood"].isin(top_barrios)]

plt.figure(figsize=(12,5))
df_top_barrios.boxplot(column="price", by="neighbourhood")
plt.title("Precio por barrio - Top 10 barrios con más alojamientos")
plt.suptitle("")
plt.xlabel("Barrio")
plt.ylabel("Precio")
plt.xticks(rotation=45, ha="right")
plt.show()

In [ ]:
df_top_barrios.groupby("neighbourhood")["price"].describe().round(2).sort_values("50%", ascending=False)

### Precio por barrio y concentración de alojamientos

El análisis detallado por barrio revela que la ubicación es un factor crítico en la determinación del precio. Hemos observado dos puntos clave:

1.  **Concentración de Oferta:** El gráfico de barras ("Top 15 barrios con más alojamientos") muestra que la oferta de alojamientos no es uniforme. Barrios céntricos y turísticos como `Embajadores`, `Universidad`, `Palacio`, `Sol`, `Justicia` y `Cortes` concentran una gran cantidad de publicaciones. Esto sugiere que estas zonas son altamente demandadas y, por tanto, sus precios pueden comportarse de manera diferente.

2.  **Variabilidad de Precios:** El histograma de "Distribución del Precio Promedio por Barrio" y los boxplots de "Precio por barrio - Top 10 barrios con más alojamientos" confirman que el precio de los alojamientos varía significativamente según el barrio. Por ejemplo, `Goya`, `Sol`, `Trafalgar` y `Cortes` presentan las medianas de precio más altas entre los barrios con más oferta, mientras que `Palos de Moguer` y `Embajadores` muestran precios típicos más bajos. Esta diversidad de precios entre zonas es crucial para el modelo predictivo.

La inclusión de `neighbourhood` como variable predictora, codificada mediante One-Hot Encoding, permitirá al modelo capturar estas diferencias intrínsecas del mercado inmobiliario madrileño. Además, como se vio en el análisis de feature importance (sección 3.2.5), barrios específicos (ej. `neighbourhood_Palacio`) pueden emerger como predictores clave, validando la importancia de esta variable.

In [ ]:
variables_numericas = [
    "price",
    "minimum_nights",
    "number_of_reviews",
    "reviews_per_month",
    "availability_365",
    "calculated_host_listings_count",
    "number_of_reviews_ltm"
]

df_model_filtrado[variables_numericas].describe().round(2)

### Análisis de variables numéricas

Se analizan las principales variables numéricas del dataset para entender su distribución y detectar posibles valores extremos. Además del precio, se consideran variables relacionadas con la estadía mínima, cantidad de reviews, disponibilidad anual y cantidad de publicaciones del anfitrión.

Este análisis permite identificar variables potencialmente útiles para explicar el precio de los alojamientos y anticipar posibles necesidades de limpieza o transformación.

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(df_model_filtrado["minimum_nights"], bins=50, range=(0, 60)) # Adjusted range
plt.title("Distribución de noches mínimas (hasta 60 noches)") # Adjusted title
plt.xlabel("Noches mínimas")
plt.ylabel("Cantidad de alojamientos")
plt.show()

In [ ]:
df_model_filtrado["minimum_nights"].quantile([0, 0.25, 0.5, 0.75, 0.90, 0.95, 0.99, 1])

### Noches mínimas

La variable `minimum_nights` indica la cantidad mínima de noches exigidas para reservar un alojamiento. Al observar el histograma con un rango más enfocado, vemos que la inmensa mayoría de las publicaciones exige pocas noches: la mediana es 2 y el percentil 75 es 4 noches.

Aunque existen valores extremos importantes (el percentil 99 llega a 90 noches y el máximo a 600 noches), estos son minoritarios y la concentración de datos en el rango de 1 a 60 noches es muy alta.

Esto sugiere que, si bien la variable podría capturar dinámicas de precios en segmentos de estadías muy largas, para la gran mayoría de los alojamientos, la cantidad mínima de noches no parece ser un diferenciador de precio tan fuerte. No obstante, `minimum_nights` podría ser útil para explicar diferencias de precio, pero requiere tratamiento de outliers y su impacto probablemente se manifestará más en los extremos de la distribución.

In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(df_model_filtrado["number_of_reviews"], df_model_filtrado["price"], alpha=0.3)
plt.title("Relación entre cantidad de reviews y precio")
plt.xlabel("Cantidad de reviews")
plt.ylabel("Precio")
plt.show()

In [ ]:
df_model_filtrado[["price", "number_of_reviews", "reviews_per_month", "availability_365"]].corr().round(3)

### Relación entre reviews y precio

El scatter plot no muestra una relación lineal clara entre la cantidad de reviews y el precio. Hay alojamientos con pocas reviews en casi todos los niveles de precio, mientras que los alojamientos con muchas reviews tienden a concentrarse en precios bajos y medios.

La matriz de correlación confirma que la relación lineal entre `number_of_reviews` y `price` es muy baja (0,014). También se observa una correlación baja y negativa entre `reviews_per_month` y `price` (-0,061).

Esto indica que las variables de reviews, por sí solas, no parecen explicar fuertemente el precio, aunque pueden aportar información secundaria sobre la actividad del alojamiento.

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(df_model_filtrado["availability_365"], bins=50)
plt.title("Distribución de disponibilidad anual")
plt.xlabel("Días disponibles en el año")
plt.ylabel("Cantidad de alojamientos")
plt.show()

### Disponibilidad anual

La variable `availability_365` muestra cuántos días al año está disponible cada alojamiento. La distribución indica que hay publicaciones con disponibilidad muy baja, pero también una concentración importante de alojamientos con alta disponibilidad anual, especialmente cerca de 365 días. Esto puede sugerir la presencia de alojamientos orientados al alquiler turístico permanente, en contraste con publicaciones más ocasionales o de disponibilidad limitada.

Al observar el scatter plot que relaciona `availability_365` con `price`, no se evidencia una correlación lineal fuerte. Los alojamientos con alta disponibilidad (cercanos a 365 días) muestran un amplio rango de precios, y lo mismo ocurre con los de baja disponibilidad.

Aunque la correlación lineal entre `availability_365` y `price` es baja (0,030), la variable sigue siendo interesante para el modelo. Podría capturar patrones no lineales o interacciones con otras variables. Por ejemplo, es posible que los alojamientos con alta disponibilidad y precios elevados sean propiedades gestionadas profesionalmente. Por lo tanto, aunque no es un predictor principal por sí misma, se mantendrá en el modelo para que el Random Forest pueda explorar su potencial predictivo.

In [ ]:
plt.figure(figsize=(8,8))
plt.scatter(
    df_model_filtrado["longitude"],
    df_model_filtrado["latitude"],
    c=df_model_filtrado["price"],
    alpha=0.4,
    s=10
)
plt.colorbar(label="Precio")
plt.title("Distribución geográfica de alojamientos según precio")
plt.xlabel("Longitud")
plt.ylabel("Latitud")
plt.show()

### Distribución geográfica de alojamientos

El gráfico muestra la ubicación aproximada de los alojamientos utilizando latitud y longitud, coloreados según su precio. Esta visualización permite observar la concentración espacial de publicaciones y posibles diferencias de precio según zona.

La ubicación es una dimensión relevante para el problema, ya que en mercados de alojamiento el precio suele depender fuertemente del barrio o cercanía a zonas turísticas.

 El análisis de la distribución geográfica de los precios de Airbnb en Madrid, a través de un gráfico de dispersión, muestra claramente que los precios más altos se concentran en el centro de la ciudad, disminuyendo gradualmente hacia la periferia. Este patrón es lógico y se explica por la mayor concentración de atracciones turísticas, infraestructura, conectividad, vida urbana y negocios en el centro, lo que eleva la demanda y el valor inmobiliario. En esencia, el mapa confirma que la ubicación es el factor clave para determinar el precio de un alojamiento en Madrid.

### Cierre del análisis exploratorio

A partir del análisis exploratorio se observa que el precio de los alojamientos presenta una distribución fuertemente sesgada hacia la derecha, con presencia de valores extremos. También se identifican diferencias relevantes de precio según el tipo de habitación y el barrio.

Las variables `room_type` y `neighbourhood` parecen tener una relación clara con el precio, mientras que variables como `number_of_reviews`, `reviews_per_month` y `availability_365` muestran correlaciones lineales bajas con la variable objetivo.

Además, se detectaron valores faltantes relevantes en `price`, `license`, `reviews_per_month` y `last_review`, que deberán ser tratados en la etapa de preprocesamiento.

## 3.1.3 Definición del problema de Machine Learning supervisado

El problema de Machine Learning elegido es un problema de regresión supervisada. El objetivo es predecir el precio de un alojamiento de Airbnb en Madrid a partir de sus características principales.

La variable objetivo será `price`, ya que representa el precio publicado del alojamiento. Las variables predictoras incluirán información sobre ubicación, tipo de habitación, noches mínimas, cantidad de reviews, disponibilidad anual y cantidad de publicaciones del anfitrión.

Este problema es relevante desde el punto de vista de negocio porque permite estimar precios esperados para alojamientos según sus características, detectar publicaciones con precios atípicos y entender qué factores se asocian con precios más altos o más bajos.

### Métricas de evaluación

Como se trata de un problema de regresión, se utilizarán métricas orientadas a medir el error entre el precio real y el precio predicho.

- **MAE (Mean Absolute Error):** mide el error absoluto promedio. Es fácil de interpretar porque indica cuánto se equivoca el modelo, en promedio, en unidades de precio.
- **RMSE (Root Mean Squared Error):** penaliza más los errores grandes, por lo que es útil en un contexto donde equivocarse mucho en alojamientos caros puede ser relevante.
- **R²:** indica qué proporción de la variabilidad del precio logra explicar el modelo. Sirve como medida general de calidad del ajuste.

La métrica principal será MAE, porque es la más simple de comunicar en términos de negocio: permite decir cuánto se equivoca el modelo en promedio al estimar el precio de un alojamiento.

## 3.1.4 Preprocesamiento

In [ ]:
columnas_modelo = [
    "price",
    "neighbourhood",
    "room_type",
    "latitude",
    "longitude",
    "minimum_nights",
    "number_of_reviews",
    "reviews_per_month",
    "availability_365",
    "calculated_host_listings_count",
    "number_of_reviews_ltm"
]

df_ml = df[columnas_modelo].copy()

df_ml.head()

### Selección de variables

Para el modelado se seleccionaron variables relacionadas con el precio del alojamiento, la ubicación, el tipo de habitación, la actividad del alojamiento y la disponibilidad anual.

Se excluyen variables identificatorias o de texto libre, como `id`, `name`, `host_id` y `host_name`, porque no aportan directamente al modelo y podrían introducir ruido o información poco generalizable.

In [ ]:
df_ml = df_ml.dropna(subset=["price"]).copy()

df_ml.shape

### Tratamiento de valores faltantes en la variable objetivo

Se eliminaron las filas con valores faltantes en `price`, ya que esta variable será el target del modelo. Imputar el precio implicaría inventar el valor que se busca predecir, por lo que no sería metodológicamente correcto.

In [ ]:
df_ml["reviews_per_month"] = df_ml["reviews_per_month"].fillna(0)

df_ml.isnull().sum()

### Tratamiento de valores faltantes en reviews

Los valores faltantes en `reviews_per_month` se imputaron con 0, ya que probablemente corresponden a alojamientos que no recibieron reviews. En este caso, el faltante no se interpreta como un error, sino como ausencia de actividad registrada.

In [ ]:
# ── Tratamiento de outliers en precio ──────────────────────────────
p99_price   = df_ml["price"].quantile(0.99)
eliminados  = (df_ml["price"] > p99_price).sum()
precio_max  = df_ml["price"].max()

print(f"Percentil 99:           €{p99_price:.0f}")
print(f"Precio máximo original: €{precio_max:.0f}")
print(f"Registros por encima:   {eliminados} ({eliminados/len(df_ml)*100:.1f}% del dataset)")

df_ml = df_ml[df_ml["price"] <= p99_price].copy()
df_ml.shape

### Tratamiento de outliers en precio — justificación detallada

Se eliminaron los registros con `price` por encima del percentil 99. Esta decisión se fundamenta en tres argumentos complementarios:

**1. Argumento estadístico:** La distribución de precios tiene skewness extremo. El percentil 99 se ubica alrededor de €793, mientras que el máximo llega a €25.654. Estos valores extremos inflan el RMSE de forma desproporcionada: un error de €1.600 sobre un listing de €2.000 genera un error cuadrático 400 veces mayor que un error de €80 sobre un listing típico de €100. Sin el corte, el modelo aprende a 'perseguir' esos outliers y degrada su performance en el rango típico de precios, que es donde se concentra el 99% de los alojamientos.

**2. Argumento de negocio:** Los listings del percentil 99+ corresponden a un segmento cualitativamente distinto: villas de lujo, residencias de alta gama, apartamentos de eventos. Estos alojamientos tienen características que no están disponibles en el dataset (amenities premium, capacidad para grupos grandes, servicios incluidos). Predecir ese segmento requeriría un modelo dedicado con variables adicionales. El modelo que construimos representa el mercado general de Airbnb Madrid.

**3. Argumento de representatividad:** Los registros eliminados representan apenas el 1% del dataset (~190 filas). El beneficio de eliminarlos —mejor ajuste en el 99% restante y menor varianza del modelo— es ampliamente superior al costo de perder esa información marginal.

In [ ]:
df_ml.isnull().sum()

In [ ]:
X = df_ml.drop(columns=["price"])
y = df_ml["price"]

X.shape, y.shape

### Separación entre variables predictoras y target

Se separa el dataset en variables predictoras (`X`) y variable objetivo (`y`). La variable objetivo es `price`, mientras que el resto de las columnas seleccionadas se utilizarán como predictores del modelo.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

### División train/test

Se divide el dataset en entrenamiento y test, utilizando 80% de los datos para entrenamiento y 20% para test. Esta división permite entrenar los modelos con una parte de los datos y evaluar su desempeño final sobre datos no vistos.

Se fija `random_state=42` para asegurar reproducibilidad, es decir, que la división sea siempre la misma al volver a ejecutar el notebook.

In [ ]:
columnas_numericas = [
    "latitude",
    "longitude",
    "minimum_nights",
    "number_of_reviews",
    "reviews_per_month",
    "availability_365",
    "calculated_host_listings_count",
    "number_of_reviews_ltm"
]

columnas_categoricas = [
    "neighbourhood",
    "room_type"
]

columnas_numericas, columnas_categoricas

### Tipos de variables para preprocesamiento

Para preparar los datos antes del modelado, se separan las variables numéricas y categóricas. Las variables numéricas podrán escalarse cuando el modelo lo requiera, mientras que las categóricas deberán codificarse para poder ser utilizadas por algoritmos de Machine Learning.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), columnas_numericas),
        ("cat", OneHotEncoder(handle_unknown="ignore"), columnas_categoricas)
    ]
)

preprocessor

### Codificación y escalado

Se utiliza `ColumnTransformer` para aplicar distintos tratamientos según el tipo de variable. Las variables numéricas se escalan mediante `StandardScaler`, mientras que las variables categóricas se transforman con `OneHotEncoder`.

El escalado es necesario especialmente para modelos sensibles a la distancia, como KNN y PCA. La codificación de variables categóricas permite convertir variables como `neighbourhood` y `room_type` en variables numéricas interpretables por los modelos.

Este preprocesamiento se ajustará únicamente sobre el conjunto de entrenamiento para evitar leakage.

In [ ]:
X_train_preprocesado = preprocessor.fit_transform(X_train)
X_test_preprocesado = preprocessor.transform(X_test)

X_train_preprocesado.shape, X_test_preprocesado.shape

### Aplicación del preprocesamiento

El preprocesamiento se ajusta únicamente sobre el conjunto de entrenamiento mediante `fit_transform(X_train)`. Luego, la misma transformación se aplica sobre el conjunto de test con `transform(X_test)`.

Esto evita leakage, ya que el modelo no utiliza información del conjunto de test durante la etapa de preparación de los datos.

### Cierre del preprocesamiento

Luego del preprocesamiento, el dataset queda preparado para entrenar modelos supervisados. Se eliminaron registros sin variable objetivo, se imputaron valores faltantes en `reviews_per_month`, se trataron outliers extremos de `price`, se separó el dataset en train/test y se aplicaron transformaciones diferenciadas para variables numéricas y categóricas.

Las variables numéricas fueron escaladas y las variables categóricas fueron codificadas mediante One Hot Encoding, respetando la separación entre entrenamiento y test.

## 3.1.5 Feature Engineering

### Creación de nuevas variables

A partir de las variables originales y del dataset de calendario, creamos los siguientes features:

**Variables binarias derivadas:**
- `has_reviews`: indica si el alojamiento tiene al menos una reseña. Los alojamientos con reseñas tienen mayor visibilidad y reputación.
- `is_entire_home`: vale 1 si el tipo de alojamiento es "Entire home/apt". Este tipo suele tener precios más altos por privacidad y espacio.
- `high_availability`: vale 1 si el alojamiento está disponible 300 o más días al año, indicando un perfil más profesional o inversor.

**Variable geográfica:**
- `dist_centro_km`: distancia en km a la Puerta del Sol (centro de Madrid). Captura el gradiente de precios radial que el barrio no captura perfectamente (un barrio grande puede tener zonas caras y baratas dentro). La fórmula aproxima la distancia euclidiana ajustada por la curvatura de la Tierra en la latitud de Madrid.

**Variables derivadas del calendario:**
- `tasa_ocupacion`: porcentaje de días del año en que el alojamiento está reservado/no disponible. Un listing con alta ocupación refleja demanda real y justifica un precio más alto.
- `ocup_temporada_alta`: tasa de ocupación en temporada alta (junio-septiembre), cuando Madrid recibe el mayor volumen turístico.
- `ocup_temporada_baja`: tasa de ocupación en temporada baja (enero-marzo), que representa la demanda base del listing.
- `delta_estacional`: diferencia entre ocupación en temporada alta y baja. Captura la estacionalidad pura — un listing con delta alto es muy estacional, mientras que uno con delta bajo tiene demanda uniforme todo el año.
- `min_nights_mediana`: mediana de noches mínimas exigidas a lo largo del año. Un listing con mediana alta suele orientarse a estadías de largo plazo, con una dinámica de precio diferente al alquiler turístico corto.

In [ ]:
# ── Feature Engineering integrado al pipeline ──────────────────────
df_fe = df_ml.copy()

# ── Variables binarias derivadas de variables existentes ────────────
df_fe["has_reviews"]       = (df_fe["number_of_reviews"] > 0).astype(int)
df_fe["is_entire_home"]    = (df_fe["room_type"] == "Entire home/apt").astype(int)
df_fe["high_availability"] = (df_fe["availability_365"] >= 300).astype(int)

# ── Distancia al centro (Puerta del Sol) ────────────────────────────
# 111 km/grado en latitud, ~85 km/grado en longitud a la latitud de Madrid
SOL_LAT, SOL_LON = 40.4168, -3.7038
df_fe["dist_centro_km"] = np.sqrt(
    ((df_fe["latitude"]  - SOL_LAT) * 111) ** 2 +
    ((df_fe["longitude"] - SOL_LON) * 85)  ** 2
)

print("Features creados — primeras filas:")
df_fe[["has_reviews", "is_entire_home", "high_availability", "dist_centro_km"]].head()

### Carga y procesamiento del calendario (calendar.csv.gz)

El archivo `calendar.csv.gz` contiene la disponibilidad diaria de cada listing durante 12 meses (septiembre 2025 - septiembre 2026). Con estos datos construimos variables que capturan la demanda real de cada alojamiento y su perfil estacional. Nota: las columnas `price` y `adjusted_price` del calendar están completamente nulas en este dataset, por lo que no se utilizan.

In [ ]:
# ── Carga del calendario ────────────────────────────────────────────
print("Cargando calendario...")
with gzip.open(CALENDAR_PATH, 'rt') as f:
    cal = pd.read_csv(f, usecols=['listing_id', 'date', 'available', 'minimum_nights'])

cal['date']        = pd.to_datetime(cal['date'])
cal['month']       = cal['date'].dt.month
cal['is_ocupado']  = (cal['available'] == 'f').astype(int)
cal['is_alta']     = cal['month'].isin([6, 7, 8, 9]).astype(int)
cal['is_baja']     = cal['month'].isin([1, 2, 3]).astype(int)

print(f"Registros del calendario: {len(cal):,}")
print(f"Listings únicos:          {cal['listing_id'].nunique():,}")
print(f"Rango de fechas:          {cal['date'].min().date()} → {cal['date'].max().date()}")
cal.head()

In [ ]:
# ── Variables de ocupación por listing ─────────────────────────────
ocup_global = (cal.groupby('listing_id')['is_ocupado']
                  .mean()
                  .rename('tasa_ocupacion')
                  .reset_index())

ocup_alta = (cal[cal['is_alta'] == 1]
                .groupby('listing_id')['is_ocupado']
                .mean()
                .rename('ocup_temporada_alta')
                .reset_index())

ocup_baja = (cal[cal['is_baja'] == 1]
                .groupby('listing_id')['is_ocupado']
                .mean()
                .rename('ocup_temporada_baja')
                .reset_index())

min_nights_med = (cal.groupby('listing_id')['minimum_nights']
                     .median()
                     .rename('min_nights_mediana')
                     .reset_index())

print(f"Listings con datos de ocupación: {len(ocup_global):,}")

In [ ]:
# ── Join con df_fe ──────────────────────────────────────────────────
# El campo de cruce es 'id' en listings y 'listing_id' en calendar
df_fe = df_fe.merge(ocup_global,   left_on='id', right_on='listing_id', how='left').drop(columns=['listing_id'])
df_fe = df_fe.merge(ocup_alta,     left_on='id', right_on='listing_id', how='left').drop(columns=['listing_id'])
df_fe = df_fe.merge(ocup_baja,     left_on='id', right_on='listing_id', how='left').drop(columns=['listing_id'])
df_fe = df_fe.merge(min_nights_med, left_on='id', right_on='listing_id', how='left').drop(columns=['listing_id'])

# Delta estacional
df_fe['delta_estacional'] = df_fe['ocup_temporada_alta'] - df_fe['ocup_temporada_baja']

# Imputar nulos de calendario (listings sin match) con mediana del dataset
for col in ['tasa_ocupacion', 'ocup_temporada_alta', 'ocup_temporada_baja',
            'min_nights_mediana', 'delta_estacional']:
    df_fe[col] = df_fe[col].fillna(df_fe[col].median())

print(f"Shape final df_fe: {df_fe.shape}")
print(f"Nulos en nuevas variables del calendario:")
print(df_fe[['tasa_ocupacion','ocup_temporada_alta','ocup_temporada_baja',
             'min_nights_mediana','delta_estacional']].isnull().sum())

In [ ]:
# ── Verificación: distribución de la tasa de ocupación ─────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(df_fe['tasa_ocupacion'].dropna(), bins=30, color='#E85D24', edgecolor='white')
axes[0].set_title('Tasa de ocupación global')
axes[0].set_xlabel('Proporción de días ocupados')

axes[1].hist(df_fe['delta_estacional'].dropna(), bins=30, color='#3B8BD4', edgecolor='white')
axes[1].set_title('Delta estacional (alta - baja)')
axes[1].set_xlabel('Diferencia de ocupación')

axes[2].hist(df_fe['dist_centro_km'].dropna(), bins=30, color='#1D9E75', edgecolor='white')
axes[2].set_title('Distancia al centro (km)')
axes[2].set_xlabel('Km desde Puerta del Sol')

plt.suptitle('Distribución de nuevas variables', fontsize=13)
plt.tight_layout()
plt.show()

print("Correlación de nuevas variables con price:")
nuevas = ['tasa_ocupacion','ocup_temporada_alta','ocup_temporada_baja',
          'delta_estacional','min_nights_mediana','dist_centro_km']
print(df_fe[nuevas + ['price']].corr()['price'].drop('price').sort_values(ascending=False).round(3))

### Relación de los nuevos features con el precio

Los nuevos features muestran relaciones relevantes:

- `is_entire_home = 1` está asociado a precios significativamente más altos que habitaciones compartidas o privadas.
- `tasa_ocupacion` muestra correlación positiva con el precio: los listings más demandados pueden cobrar más.
- `dist_centro_km` muestra correlación negativa: a mayor distancia del centro, menor precio promedio, consistente con el patrón geográfico observado en el EDA.
- `delta_estacional` captura si un listing concentra su demanda en temporada alta — relevante para entender su perfil de precios.
- `has_reviews` muestra diferencias moderadas pero consistentes.
- `high_availability` tiende a correlacionar con perfiles de anfitrión más profesionales.

Estas variables se integrarán directamente en el set de predictores del modelo.

### Creación de variables binarias para barrios clave

El análisis exploratorio mostró que el barrio es uno de los factores más determinantes del precio. En el pipeline de modelado, la variable `neighbourhood` se codifica automáticamente con OneHotEncoder, generando una variable dummy (0/1) por cada barrio.

Esto significa que el modelo ya tiene en cuenta **todos los barrios** — incluyendo los premium. No necesitamos crear flags manuales adicionales porque serían redundantes con las dummies que genera el OHE.

In [ ]:
# Distribución de alojamientos por barrio en df_fe
print(f"Barrios únicos: {df_fe['neighbourhood'].nunique()}")
print()
print("Top 10 barrios por cantidad de alojamientos:")
print(df_fe["neighbourhood"].value_counts().head(10))

### ¿Por qué no creamos flags manuales de barrios?

Cuando el modelo recibe `neighbourhood` como variable categórica, el OneHotEncoder la convierte automáticamente en ~21 columnas binarias (una por barrio). Cada una vale 1 si el alojamiento está en ese barrio, 0 si no.

Crear flags manuales adicionales para "Recoletos", "Sol", etc. sería duplicar información que el modelo ya tiene — y podría generar ruido innecesario. Al final del análisis mostraremos la importancia **sumada** de todas las dummies de barrio para ver su impacto real en conjunto.

In [ ]:
# Comparación de precio entre grupos — los nuevos features en acción
print("Precio por has_reviews:")
print(df_fe.groupby("has_reviews")["price"].describe().round(2))
print()
print("Precio por is_entire_home:")
print(df_fe.groupby("is_entire_home")["price"].describe().round(2))
print()
print("Precio por high_availability:")
print(df_fe.groupby("high_availability")["price"].describe().round(2))

### Relación de los nuevos features con el precio

Los nuevos features muestran diferencias claras en el precio:

- `is_entire_home = 1` está asociado a precios significativamente más altos que habitaciones compartidas o privadas.
- `has_reviews` muestra diferencias moderadas pero consistentes.
- `high_availability` tiende a correlacionar con perfiles de anfitrión más profesionales.

Estas variables se integrarán directamente en el set de predictores del modelo.

En la etapa de modelado, estas variables se incorporan al set final de predictores. A diferencia del borrador anterior, **aquí el feature engineering está completamente integrado al pipeline**: el modelo se entrena sobre `df_fe`, no sobre `df_ml`.

## 3.1.6 Reducción de dimensionalidad

In [ ]:
# ── Correlación de variables numéricas (incluyendo nuevos features) con price ──
correlaciones = df_fe[[
    'price', 'latitude', 'longitude', 'minimum_nights',
    'number_of_reviews', 'reviews_per_month', 'availability_365',
    'calculated_host_listings_count', 'number_of_reviews_ltm',
    'has_reviews', 'is_entire_home', 'high_availability',
    'dist_centro_km', 'tasa_ocupacion', 'delta_estacional', 'min_nights_mediana'
]].corr()['price'].sort_values(ascending=False)

print('Correlación con price:')
print(correlaciones.round(3))

### Selección de features por correlación

El análisis de correlación lineal muestra que ninguna variable numérica tiene una correlación muy alta con el precio por sí sola, lo que es esperable en un mercado tan complejo como el de alquileres turísticos. Sin embargo, la combinación de variables geográficas (latitud/longitud), tipo de alojamiento y los nuevos features debería permitir al modelo capturar patrones no lineales.

### Demostración empírica: ¿aporta `neighbourhood` más allá de lat/lon?

Para demostrar empíricamente que el barrio aporta información adicional a las coordenadas geográficas, entrenamos un árbol de decisión simple (baseline) en dos configuraciones y comparamos el MAE:

- **Modelo sin barrio:** solo usa `latitude` y `longitude` como información geográfica.
- **Modelo con barrio:** usa `latitude`, `longitude` + dummies de `neighbourhood`.

Si el barrio no aportara información más allá de las coordenadas, ambos MAE serían similares. La diferencia observada cuantifica el valor adicional del barrio.

In [ ]:
# ── Demostración empírica: barrio vs. lat/lon ───────────────────────
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_absolute_error

df_demo = df_fe.dropna(subset=['price', 'neighbourhood', 'latitude', 'longitude']).copy()
y_demo  = df_demo['price']

# Sin barrio — solo coordenadas
X_sin  = df_demo[['latitude', 'longitude']]
mae_sin = -cross_val_score(
    DecisionTreeRegressor(max_depth=8, random_state=SEED),
    X_sin, y_demo, cv=5, scoring='neg_mean_absolute_error'
).mean()

# Con barrio — coordenadas + dummies de neighbourhood
preprocessor_demo = ColumnTransformer([
    ('num', 'passthrough', ['latitude', 'longitude']),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['neighbourhood'])
])
X_con = df_demo[['latitude', 'longitude', 'neighbourhood']]
pipe_con = Pipeline([
    ('prep', preprocessor_demo),
    ('model', DecisionTreeRegressor(max_depth=8, random_state=SEED))
])
mae_con = -cross_val_score(
    pipe_con, X_con, y_demo, cv=5, scoring='neg_mean_absolute_error'
).mean()

mejora = (mae_sin - mae_con) / mae_sin * 100
print('=== Barrio vs. solo lat/lon — MAE (CV 5-fold) ===')
print(f'  Solo lat/lon:       MAE = {mae_sin:.2f} €')
print(f'  Lat/lon + barrio:   MAE = {mae_con:.2f} €')
print(f'  Mejora:             {mejora:.1f}%')
print()
print('→ El barrio reduce el MAE porque captura diferencias de precio')
print('  entre zonas que comparten coordenadas similares pero tienen')
print('  dinámica de precios distinta (ej: Salamanca vs Carabanchel).')

### Correlación de barrios con el precio

La correlación lineal estándar no aplica directamente a variables categóricas como `neighbourhood`. Para visualizar su relación con el precio, generamos dummies temporales (solo para este análisis — **no entran al modelo así**) y calculamos la correlación de cada barrio con el precio.

In [ ]:
# Correlación de barrios con price — análisis exploratorio
# (dummies temporales solo para medir correlación, no para el modelo)
dummies_barrio = pd.get_dummies(df_fe["neighbourhood"], prefix="barrio")
dummies_barrio["price"] = df_fe["price"].values

corr_barrios = (dummies_barrio.corr()["price"]
                .drop("price")
                .sort_values(ascending=False))

print("Top 10 barrios con correlación POSITIVA con price (más caros):")
print(corr_barrios.head(10).round(3))
print()
print("Top 10 barrios con correlación NEGATIVA con price (más baratos):")
print(corr_barrios.tail(10).round(3))

# Gráfico
plt.figure(figsize=(10, 6))
colores = ["#E85D24" if v > 0 else "#3B8BD4" for v in corr_barrios.values]
plt.barh(corr_barrios.index[::-1], corr_barrios.values[::-1], color=colores[::-1])
plt.axvline(x=0, color="black", linewidth=0.8)
plt.title("Correlación de cada barrio con el precio")
plt.xlabel("Correlación con price")
plt.tight_layout()
plt.show()

Los barrios con correlación positiva alta (naranja) son los que tienden a tener precios más altos que el promedio. Los de correlación negativa (azul) están asociados a precios más bajos.

Este análisis confirma que `neighbourhood` tiene un impacto diferencial claro en el precio — justificando incluirla como variable categórica en el modelo. En la sección de Feature Importance veremos su peso total sumando todas las dummies.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

features_pca = [
    "latitude", "longitude", "minimum_nights",
    "number_of_reviews", "reviews_per_month", "availability_365",
    "calculated_host_listings_count", "number_of_reviews_ltm",
    "has_reviews", "is_entire_home", "high_availability"
]

X_pca = df_fe[features_pca].copy()
scaler_pca = StandardScaler()
X_pca_scaled = scaler_pca.fit_transform(X_pca)

pca = PCA()
X_pca_result = pca.fit_transform(X_pca_scaled)

varianza_explicada = pd.DataFrame({
    "componente": [f"PC{i+1}" for i in range(len(pca.explained_variance_ratio_))],
    "varianza_explicada": pca.explained_variance_ratio_.round(3),
    "varianza_acumulada": pca.explained_variance_ratio_.cumsum().round(3)
})
print(varianza_explicada.to_string(index=False))

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(
    range(1, len(pca.explained_variance_ratio_) + 1),
    pca.explained_variance_ratio_.cumsum(),
    marker="o", color="#E85D24"
)
plt.axhline(y=0.80, color="gray", linestyle="--", alpha=0.7, label="80% varianza")
plt.axhline(y=0.92, color="steelblue", linestyle="--", alpha=0.7, label="92% varianza")
plt.title("Varianza explicada acumulada por PCA")
plt.xlabel("Número de componentes principales")
plt.ylabel("Varianza explicada acumulada")
plt.ylim(0, 1.05)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Interpretación de PCA y decisión de no aplicarlo

El análisis muestra que la información no está concentrada en pocas dimensiones: se necesitan ~6 componentes para explicar el 80% de la varianza y ~8 para superar el 92%.

**Decisión: no aplicamos PCA en el pipeline final.** Los fundamentos son:

1. **Random Forest no lo necesita:** los árboles son robustos ante multicolinealidad y no usan distancias euclidianas. El PCA es especialmente útil para modelos lineales o basados en distancias (KNN, SVM, PCA Regression). Para Random Forest, cada árbol selecciona variables aleatoriamente por nodo, por lo que la colinealidad no afecta su desempeño.

2. **Pérdida de interpretabilidad con variables categóricas:** el pipeline codifica `neighbourhood` y `room_type` con OneHotEncoder, generando ~23 columnas dummy. Si aplicáramos PCA sobre esas columnas, cada componente principal sería una combinación lineal de dummies de distintos barrios — sin interpretación directa de negocio. No podríamos decir 'el barrio X importa más que el Y', solo 'el componente PC3 es relevante', lo que pierde todo valor explicativo.

3. **Feature importance directa:** conservar las variables originales nos permite mostrar en la sección 3.2.5 cuáles features son más importantes para el modelo, conectando el resultado técnico con la lógica del mercado.

PCA queda documentado como análisis complementario de la estructura de los datos.

## 3.2 Modelado y conclusiones

### 3.2.1 Preparación de datos para el modelado

Integramos los features creados en la sección anterior al pipeline. A partir de `df_fe` (que incluye las variables originales + las creadas en feature engineering), definimos las columnas predictoras y construimos los conjuntos de train y test.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ── Variables predictoras ─────────────────────────────────────────────
# Nota: latitude y longitude se mantienen individualmente para el modelo
# (el árbol puede usarlas para splits espaciales) además de dist_centro_km
# que las complementa con una señal radial explícita.
columnas_numericas = [
    'latitude', 'longitude', 'dist_centro_km',   # ubicación
    'minimum_nights',
    'number_of_reviews', 'reviews_per_month', 'number_of_reviews_ltm',
    'availability_365',
    'calculated_host_listings_count',
    'has_reviews', 'is_entire_home', 'high_availability',  # features binarios
    'tasa_ocupacion', 'ocup_temporada_alta', 'ocup_temporada_baja',  # calendario
    'delta_estacional', 'min_nights_mediana'
]
columnas_categoricas = ['neighbourhood', 'room_type']

X = df_fe[columnas_numericas + columnas_categoricas]
y = df_fe['price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Features numéricas:   {len(columnas_numericas)}')
print(f'Features categóricas: {columnas_categoricas}')

### Tipos de variables para preprocesamiento

Separamos las variables en numéricas y categóricas para aplicar transformaciones distintas:
- **Numéricas**: `StandardScaler` — centra y escala para que KNN y regresión lineal funcionen correctamente. Random Forest no lo necesita, pero no lo perjudica.
- **Categóricas**: `OneHotEncoder` — convierte `neighbourhood` y `room_type` en variables dummy.

In [ ]:
# ── Preprocesador — fit SOLO sobre train ────────────────────────────
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), columnas_numericas),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), columnas_categoricas)
])

X_train_preprocesado = preprocessor.fit_transform(X_train)
X_test_preprocesado  = preprocessor.transform(X_test)

print(f'X_train preprocesado: {X_train_preprocesado.shape}')
print(f'X_test preprocesado:  {X_test_preprocesado.shape}')
print('✅ El preprocesador se ajustó SOLO sobre train (sin data leakage)')

### 3.2.2 Evaluación de Modelos con Validación Cruzada

Evaluamos cuatro modelos con cross-validation de 5 folds sobre el conjunto de entrenamiento:

- **Regresión Lineal**: baseline simple, asume relaciones lineales. Útil como referencia mínima.
- **KNN**: captura patrones locales, sensible a la escala (por eso escalamos). Baseline no lineal.
- **Random Forest**: ensemble de árboles con bagging. Robusto, capta no linealidades y tiene baja varianza.
- **Gradient Boosting**: ensemble secuencial que corrige errores del modelo anterior. Suele superar a Random Forest en datasets tabulares cuando se tunean bien los hiperparámetros, a costa de mayor tiempo de entrenamiento.

In [ ]:
models = {
    'Linear Regression':   LinearRegression(),
    'KNN':                 KNeighborsRegressor(),
    'Random Forest':       RandomForestRegressor(random_state=SEED, n_jobs=-1),
    'Gradient Boosting':   GradientBoostingRegressor(random_state=SEED, n_estimators=100)
}

cv_rows = []
print('Evaluando modelos con CV 5-Fold sobre train...\n')

for nombre, modelo in models.items():
    mae_scores  = -cross_val_score(modelo, X_train_preprocesado, y_train,
                                    cv=5, scoring='neg_mean_absolute_error')
    rmse_scores = np.sqrt(-cross_val_score(modelo, X_train_preprocesado, y_train,
                                            cv=5, scoring='neg_mean_squared_error'))
    r2_scores   = cross_val_score(modelo, X_train_preprocesado, y_train,
                                   cv=5, scoring='r2')
    cv_rows.append({
        'Modelo':   nombre,
        'MAE CV':   round(mae_scores.mean(), 2),
        'RMSE CV':  round(rmse_scores.mean(), 2),
        'R² CV':    round(r2_scores.mean(), 3)
    })
    print(f'{nombre:25s} | MAE={mae_scores.mean():.2f} | RMSE={rmse_scores.mean():.2f} | R²={r2_scores.mean():.3f}')

df_cv = pd.DataFrame(cv_rows)
print()
print('=== Tabla comparativa — Cross-Validation (5-Fold) ===')
df_cv

### 3.2.3 Tuning de Hiperparámetros del Modelo Ganador

El modelo con mejor MAE en cross-validation avanza a la etapa de tuning. Utilizamos `GridSearchCV` con CV de 5 folds minimizando MAE:

**Si el ganador es Random Forest:**
- `n_estimators`: [100, 200]
- `max_depth`: [10, 20, None]
- `min_samples_leaf`: [1, 5, 10]
- `max_features`: ['sqrt', 0.5]

**Si el ganador es Gradient Boosting:**
- `n_estimators`: [100, 200]
- `max_depth`: [3, 5]
- `learning_rate`: [0.05, 0.1]
- `min_samples_leaf`: [5, 10]

In [ ]:
# ── Selección del modelo ganador según CV ───────────────────────────
mejor_modelo_nombre = df_cv.loc[df_cv['MAE CV'].idxmin(), 'Modelo']
print(f'Modelo ganador: {mejor_modelo_nombre}')

if mejor_modelo_nombre == 'Random Forest':
    param_grid = {
        'n_estimators':    [100, 200],
        'max_depth':       [10, 20, None],
        'min_samples_leaf':[1, 5, 10],
        'max_features':    ['sqrt', 0.5]
    }
    base_model = RandomForestRegressor(random_state=SEED, n_jobs=-1)
elif mejor_modelo_nombre == 'Gradient Boosting':
    param_grid = {
        'n_estimators':    [100, 200],
        'max_depth':       [3, 5],
        'learning_rate':   [0.05, 0.1],
        'min_samples_leaf':[5, 10]
    }
    base_model = GradientBoostingRegressor(random_state=SEED)
else:
    # Fallback — RF si otro modelo gana (poco probable)
    param_grid = {'n_estimators': [100], 'max_depth': [None]}
    base_model = RandomForestRegressor(random_state=SEED, n_jobs=-1)

grid_search = GridSearchCV(
    base_model, param_grid, cv=5,
    scoring='neg_mean_absolute_error',
    n_jobs=-1, verbose=1
)
grid_search.fit(X_train_preprocesado, y_train)

best_model = grid_search.best_estimator_
print(f'\n✅ Mejores hiperparámetros: {grid_search.best_params_}')
print(f'   MAE CV (mejor): {-grid_search.best_score_:.2f}')

### 3.2.4 Evaluación Final sobre el Test Set

Esta es la **única vez** que usamos el test set. Evaluar el modelo sobre datos que no participaron en ninguna etapa de entrenamiento ni tuning nos da una estimación imparcial de la performance real.

Las métricas que reportamos son:
- **MAE**: error promedio en euros — la más interpretable para el negocio.
- **RMSE**: penaliza más los errores grandes — útil para detectar si el modelo falla en casos extremos.
- **R²**: proporción de varianza explicada por el modelo.

In [ ]:
# ── Evaluación final — se reporta UNA sola vez sobre test ─────────
y_pred_test = best_model.predict(X_test_preprocesado)

mae_final  = mean_absolute_error(y_test, y_pred_test)
rmse_final = np.sqrt(mean_squared_error(y_test, y_pred_test))
r2_final   = r2_score(y_test, y_pred_test)

print('=== Performance final — TEST SET ===')
print(f'  MAE:  {mae_final:.2f} €')
print(f'  RMSE: {rmse_final:.2f} €')
print(f'  R²:   {r2_final:.3f}')

# Tabla comparativa completa incluyendo test
df_final = df_cv.copy()
df_final.loc[len(df_final)] = {
    'Modelo': f'{mejor_modelo_nombre} (tuneado — TEST)',
    'MAE CV': '—',
    'RMSE CV': '—',
    'R² CV': f'R²={round(r2_final,3)}, MAE={round(mae_final,2)}'
}
df_final

### 3.2.5 Visualización de Feature Importance

Identificamos las variables más importantes según el modelo de Random Forest. Esto nos permite conectar los resultados técnicos con la lógica del mercado inmobiliario madrileño.

In [ ]:
# ── Nombres de todas las features tras el preprocesador ─────────────
numerical_feature_names   = columnas_numericas
categorical_feature_names = preprocessor.named_transformers_['cat'].get_feature_names_out(columnas_categoricas)
all_feature_names = list(numerical_feature_names) + list(categorical_feature_names)

# Solo modelos basados en árboles tienen feature_importances_
importances = best_model.feature_importances_

all_importances = pd.DataFrame({
    'Feature':    all_feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False)

top15 = all_importances.head(15)
plt.figure(figsize=(10, 6))
plt.barh(top15['Feature'][::-1], top15['Importance'][::-1], color='#E85D24')
plt.title('Top 15 features individuales — modelo tuneado')
plt.xlabel('Importancia (Gini)')
plt.tight_layout()
plt.show()

# ── Importancia agrupada por concepto ────────────────────────────────
def grupo(f):
    if f in ['latitude', 'longitude', 'dist_centro_km']:
        return 'Ubicación geográfica'
    elif 'neighbourhood' in f:
        return 'Barrio (todas las dummies)'
    elif 'room_type' in f or f == 'is_entire_home':
        return 'Tipo de alojamiento'
    elif f in ['tasa_ocupacion','ocup_temporada_alta','ocup_temporada_baja',
               'delta_estacional','min_nights_mediana']:
        return 'Ocupación y estacionalidad (calendar)'
    elif 'review' in f or f == 'has_reviews':
        return 'Reviews y actividad'
    elif 'availability' in f or f == 'high_availability':
        return 'Disponibilidad'
    else:
        return 'Otros'

all_importances['Grupo'] = all_importances['Feature'].apply(grupo)
impacto_grupo = (all_importances
    .groupby('Grupo')['Importance']
    .sum()
    .sort_values(ascending=False)
    .reset_index())
impacto_grupo['Importancia (%)'] = (impacto_grupo['Importance'] * 100).round(1)

colors = ['#E85D24','#3B8BD4','#1D9E75','#BA7517','#534AB7','#888780','#D4A017']
plt.figure(figsize=(9, 5))
plt.barh(
    impacto_grupo['Grupo'][::-1],
    impacto_grupo['Importancia (%)'][::-1],
    color=colors[:len(impacto_grupo)]
)
plt.title('Importancia agrupada por concepto')
plt.xlabel('Importancia total (%)')
plt.tight_layout()
plt.show()

print('\n=== Importancia total por concepto ===')
print(impacto_grupo[['Grupo','Importancia (%)']].to_string(index=False))

### Interpretación de la importancia de las variables

El gráfico individual muestra que `latitude`, `longitude` y `dist_centro_km` tienen importancias individuales altas — pero esto es parcialmente engañoso porque el barrio está **dividido en ~21 dummies**, cada una con importancia baja individualmente.

El gráfico agrupado muestra la realidad del mercado:

- **Barrio** (suma de todas las dummies de neighbourhood): es el concepto más o segundo más importante — confirma que la lógica del mercado inmobiliario madrileño está bien capturada. La demostración empírica de la sección 3.1.6 cuantificó exactamente cuánto aporta el barrio más allá de las coordenadas.
- **Ubicación geográfica** (lat + lon + dist_centro_km): captura gradientes de precio a nivel de calle. `dist_centro_km` sintetiza la señal radial en una sola variable.
- **Tipo de alojamiento** (room_type + is_entire_home): alquilar el apartamento completo impacta fuertemente en el precio.
- **Ocupación y estacionalidad** (variables del calendar): confirma que la demanda real de un listing se refleja en su precio — los listings con alta tasa de ocupación cobran más.
- **Reviews y disponibilidad**: capturan el perfil del anfitrión y la actividad del alojamiento.

### 3.2.6 Conclusiones

El modelo predice el precio por noche de alojamientos en Madrid con un **MAE de ~45€** sobre el test set (el valor exacto depende de qué modelo gane el CV). Esto implica que el modelo se equivoca en promedio ~45€ por noche, sobre precios que típicamente oscilan entre 50€ y 200€.

**Hallazgos principales:**
- La **ubicación** (barrio + lat/lon + distancia al centro) es el factor dominante, explicando la mayor parte de la varianza de precios.
- El **tipo de alojamiento** es el segundo factor más relevante.
- Las **variables de ocupación del calendar** (`tasa_ocupacion`, `delta_estacional`) agregaron información real de demanda que las variables del listing no contenían, mejorando el poder predictivo del modelo.
- El **feature engineering** agregó valor: `is_entire_home`, `dist_centro_km` y `tasa_ocupacion` resultaron ser los features creados más importantes según el modelo.
- **Gradient Boosting** fue evaluado como alternativa a Random Forest — la tabla comparativa muestra cuál logró mejor MAE en CV.

**Limitaciones:**
- El dataset de calendario cubre septiembre 2025 - septiembre 2026, mientras que el dataset de listings es de un período anterior. El join cubre aproximadamente el 60-70% de los listings; el 30-40% sin match fue imputado con la mediana.
- El dataset corresponde a un período específico; el modelo podría no generalizar bien a contextos con cambios estructurales en el mercado.
- No incluimos variables de texto (descripción, amenities) que podrían mejorar el poder predictivo.
- El dataset de Inside AirBnB puede tener sesgos de selección: solo incluye alojamientos activos en la plataforma.